In [2]:
# ── Cell 1: Install dependencies ──────────────────────────────────
!pip install -q accelerate pillow scikit-learn openpyxl numpy qwen-vl-utils
!pip install -q git+https://github.com/huggingface/transformers accelerate
print("Install complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 79.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Install complete.


In [3]:
# ── Cell 2: Mount Google Drive & set paths ───────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os

# ── UPDATE THESE TWO PATHS ────────────────────────────────────────
BASE_DIR   = "/content/drive/MyDrive/DIPA2/dataset"
IMAGE_DIR  = os.path.join(BASE_DIR, "images")        # folder of .jpg images
CSV_PATH   = os.path.join(BASE_DIR, "annotations.csv")  # flat annotations CSV
OUTPUT_DIR = "/content/drive/MyDrive/DIPA2/qwen results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Images : {IMAGE_DIR}")
print(f"CSV    : {CSV_PATH}")
print(f"Output : {OUTPUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Images : /content/drive/MyDrive/DIPA2/dataset/images
CSV    : /content/drive/MyDrive/DIPA2/dataset/annotations.csv
Output : /content/drive/MyDrive/DIPA2/qwen results


In [4]:
# ── Cell 3: Imports ───────────────────────────────────────────────
import os, json, re, time, random, gc
from pathlib import Path
from PIL import Image
from collections import Counter
from typing import List, Dict, Set, Tuple
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print("VRAM    :", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : NVIDIA A100-SXM4-40GB
VRAM    : 42.4 GB


In [5]:
# ── Cell 4: Configuration ─────────────────────────────────────────
MODEL_ID     = "Qwen/Qwen3-VL-8B-Instruct"
MODEL_NAME   = "qwen/qwen3-vl-8b"
DATASET_NAME = "DIPA2"
DATASET_SLUG = "dipa2"

NUM_RUNS         = 3
TEMPERATURES     = [0.1, 1.0]
MAX_IMAGE_PX     = 1024
MAX_TOKENS       = {"task3": 150}   # Task 3 only — DIPA2 has no safe images
RUN_SEEDS        = [0, 42, 84]
CHECKPOINT_EVERY = 10               # save checkpoint every N images

# Resume from checkpoint if a previous run crashed
# Set RESUME=True and point RESUME_PATH to the checkpoint JSON
RESUME       = False
RESUME_PATH  = ""   # e.g. "/content/drive/MyDrive/DIPA2/results/checkpoints/ckpt_task3_temp0_1_50imgs_....json"

print(f"Model      : {MODEL_ID}")
print(f"Dataset    : {DATASET_NAME}  (Task 3 only — all images are private)")
print(f"Temps      : {TEMPERATURES}")
print(f"Runs/image : {NUM_RUNS}  |  Seeds: {RUN_SEEDS}")
print(f"Resume     : {RESUME}")

Model      : Qwen/Qwen3-VL-8B-Instruct
Dataset    : DIPA2  (Task 3 only — all images are private)
Temps      : [0.1, 1.0]
Runs/image : 3  |  Seeds: [0, 42, 84]
Resume     : False


In [6]:
# ── Cell 5: Privacy Taxonomy (paper Table 7) ─────────────────────
PRIVACY_TAXONOMY = {
    "Biometric Data":                {"examples": ["face","fingerprints","audio","iris","gait"]},
    "Children Images":               {"examples": ["school events","playgrounds"]},
    "Financial Information":         {"examples": ["credit cards","checks","receipts"]},
    "HIPAA Data":                    {"examples": ["medical records","prescriptions","health devices","disabilities"]},
    "Legal Identifiers":             {"examples": ["names","IDs","passports","addresses"]},
    "Digital Identifiers":           {"examples": ["email","phone number","passwords","computer screen content"]},
    "Personal Metadata (Demographics)":{"examples": ["gender","race","age","beliefs","occupation"]},
    "GPS Data":                      {"examples": ["gps data","live location"]},
    "Vehicle Information":           {"examples": ["license plates","vehicle ownership"]},
    "Nudity":                        {"examples": ["nudity","explicit content","adult imagery"]},
    "Violent/Unlawful Actions":      {"examples": ["criminal acts","weapons","vandalism","cigarettes"]},
    "Personal Context":              {"examples": ["pets","home interior","family gatherings","personal items"]},
    "Location Identifiers":          {"examples": ["location photos","landmarks"]},
    "Background Individuals":        {"examples": ["passerby","bystanders","not clearly visible individuals"]},
}
VALID_CATEGORIES = set(PRIVACY_TAXONOMY.keys())

# DIPA2 covers only these 8 taxonomy categories (paper Table 8)
# Paper names in parentheses: Legal Sensitivity Info, License Plates, Personal Life
DIPA2_ACTIVE_CATEGORIES = {
    "Biometric Data",           # paper: Biometric Data
    "Digital Identifiers",      # paper: Digital Identifiers
    "Legal Identifiers",        # paper: Legal Identifiers
    "Violent/Unlawful Actions", # paper: Legal Sensitivity Info
    "Vehicle Information",      # paper: License Plates
    "Location Identifiers",     # paper: Location Identifiers
    "Background Individuals",   # paper: Background Individuals
    "Personal Context",         # paper: Personal Life
}

print(f"Full taxonomy  : {len(PRIVACY_TAXONOMY)} categories")
print(f"DIPA2 active   : {len(DIPA2_ACTIVE_CATEGORIES)} categories")
print(f"Active cats    : {sorted(DIPA2_ACTIVE_CATEGORIES)}")

Full taxonomy  : 14 categories
DIPA2 active   : 8 categories
Active cats    : ['Background Individuals', 'Biometric Data', 'Digital Identifiers', 'Legal Identifiers', 'Location Identifiers', 'Personal Context', 'Vehicle Information', 'Violent/Unlawful Actions']


In [7]:
# ── Cell 6: DIPA2 → Taxonomy mapping (paper Table 8) ─────────────
# Keys are lowercase DIPA2 DIPACategory values from annotations.csv
# "Others" is excluded per paper — spans multiple concepts, not mappable

DIPA2_TO_TAXONOMY = {
    # Biometric Data — DIPA2: Person, Finger
    "person":               "Biometric Data",
    "finger":               "Biometric Data",
    # Digital Identifiers — DIPA2: Machine, Screen, Electronic Devices
    "machine":              "Digital Identifiers",
    "screen":               "Digital Identifiers",
    "electronic devices":   "Digital Identifiers",
    "electronic":           "Digital Identifiers",
    # Legal Identifiers — DIPA2: Identity, Printed Material
    "identity":             "Legal Identifiers",
    "printed material":     "Legal Identifiers",
    "printed materials":    "Legal Identifiers",  # both spellings appear in data
    # Vehicle Information — DIPA2: Vehicle Plate
    "vehicle plate":        "Vehicle Information",
    "license plate":        "Vehicle Information",
    # Violent/Unlawful Actions — DIPA2: Cigarettes
    "cigarettes":           "Violent/Unlawful Actions",
    "cigarette":            "Violent/Unlawful Actions",
    # Personal Context — DIPA2: Clothing, Book, Table, Toy, Home interior,
    #                           Cosmetics, Musical instrument, Accessory, Pet, Food
    "clothing":             "Personal Context",
    "book":                 "Personal Context",
    "table":                "Personal Context",
    "toy":                  "Personal Context",
    "home interior":        "Personal Context",
    "cosmetics":            "Personal Context",
    "musical instrument":   "Personal Context",
    "accessory":            "Personal Context",
    "pet":                  "Personal Context",
    "food":                 "Personal Context",
    # Location Identifiers — DIPA2: Place Identifier, Scenery
    "place identifier":     "Location Identifiers",
    "scenery":              "Location Identifiers",
    # Background Individuals — DIPA2: Photo (photos of people)
    "photo":                "Background Individuals",
}

print(f"DIPA2 mapping entries: {len(DIPA2_TO_TAXONOMY)}")

# Verify all active categories are reachable
reachable = set(DIPA2_TO_TAXONOMY.values())
print(f"Reachable active cats: {reachable & DIPA2_ACTIVE_CATEGORIES}")
missing = DIPA2_ACTIVE_CATEGORIES - reachable
if missing:
    print(f"WARNING — unreachable: {missing}")
else:
    print("All 8 active categories are reachable via mapping ✓")

DIPA2 mapping entries: 26
Reachable active cats: {'Background Individuals', 'Violent/Unlawful Actions', 'Digital Identifiers', 'Legal Identifiers', 'Personal Context', 'Location Identifiers', 'Biometric Data', 'Vehicle Information'}
All 8 active categories are reachable via mapping ✓


In [8]:
# ── Cell 7: Dataset loader (DIPA2 from annotations.csv) ───────────
# Selection: unanimous annotator agreement (4/4 annotators flagged privacy)
# This gives the cleanest possible ground truth — zero ambiguity.
# Result: 265 images, 412 image-category annotation pairs across 8 categories.
#
# Note: Tsaprazlis et al. report 322/332 images with undocumented selection
# criteria. Exact replication is not possible; unanimous selection is more
# methodologically rigorous.

def load_dipa2_from_csv(csv_path, image_dir):
    df = pd.read_csv(csv_path)
    print(f"Loaded CSV: {len(df)} rows, {df['imagePath'].nunique()} unique images")

    # Step 1: Find images where ALL 4 annotators flagged privacy-threatening content
    # (any DIPACategory that is not 'Others' = annotator said this object is private)
    privacy_per_image = (
        df[df['DIPACategory'].str.strip().str.lower() != 'others']
        .groupby('imagePath').size()
    )
    unanimous_images = set(privacy_per_image[privacy_per_image >= 4].index)
    print(f"Unanimous images (4/4 annotators agree): {len(unanimous_images)}")

    # Step 2: Build taxonomy labels for each unanimous image
    sub = df[
        df['imagePath'].isin(unanimous_images) &
        (df['DIPACategory'].str.strip().str.lower() != 'others')
    ].copy()
    sub['taxonomy'] = sub['DIPACategory'].str.strip().str.lower().map(DIPA2_TO_TAXONOMY)
    sub = sub[sub['taxonomy'].notna()]

    # Deduplicate: one row per image+taxonomy (multiple annotators flagged same thing)
    tax_per_image = (
        sub.groupby('imagePath')['taxonomy']
        .apply(set)
        .reset_index()
    )

    # Step 3: Build sample list, resolve image paths
    samples = []
    missing = 0
    for _, row in tax_per_image.iterrows():
        img_id   = row['imagePath']
        img_stem = Path(img_id).stem
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png', '']:
            c = Path(image_dir) / f"{img_stem}{ext}"
            if c.exists(): img_path = c; break
        if img_path is None:
            c = Path(image_dir) / img_id
            if c.exists(): img_path = c
        if img_path is None:
            missing += 1
            continue
        samples.append({
            "id":              img_stem,
            "image_path":      str(img_path),
            "is_private":      True,
            "taxonomy_labels": row['taxonomy'],   # set of taxonomy category strings
        })

    print(f"Loaded  : {len(samples)} samples  ({missing} not found in images folder)")
    print(f"All images are private (DIPA2 — no safe class, unanimous agreement filter)")

    total_pairs = sum(len(s['taxonomy_labels']) for s in samples)
    print(f"Total image-category annotation pairs: {total_pairs}")

    cat_counts = Counter()
    for s in samples:
        for c in s['taxonomy_labels']: cat_counts[c] += 1
    print("\nTaxonomy label distribution:")
    for cat, cnt in sorted(cat_counts.items(), key=lambda x: -x[1]):
        print(f"  {cat:45s} {cnt:4d}")
    return samples

dataset = load_dipa2_from_csv(CSV_PATH, IMAGE_DIR)
print(f"\nDataset ready: {len(dataset)} images")

Loaded CSV: 5920 rows, 1289 unique images
Unanimous images (4/4 annotators agree): 265
Loaded  : 265 samples  (0 not found in images folder)
All images are private (DIPA2 — no safe class, unanimous agreement filter)
Total image-category annotation pairs: 412

Taxonomy label distribution:
  Biometric Data                                 204
  Personal Context                               122
  Digital Identifiers                             35
  Vehicle Information                             21
  Location Identifiers                            16
  Legal Identifiers                                8
  Violent/Unlawful Actions                         4
  Background Individuals                           2

Dataset ready: 265 images


In [9]:
# ── Cell 8: Load Qwen3-VL-8B in full bfloat16 (A100 — no quantization needed) ─
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Loading model in bfloat16 (full precision)...")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated()/1e9
    total = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f"VRAM used: {used:.1f} GB / {total:.1f} GB")
print("Model ready.")

Loading processor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

VRAM used: 17.5 GB / 42.4 GB
Model ready.


In [10]:
# ── Cell 9: Task 3 Prompt (Safe EXCLUDED — all DIPA2 images private) ─
# Per paper: since all DIPA2 images are private, the "Safe" answer option
# is removed from the prompt. The model must always identify categories.
# Taxonomy is listed WITHOUT numbering to prevent the model from outputting
# index numbers instead of category names (known failure mode).

def get_taxonomy_string():
    lines = ["Taxonomy:"]
    for cat, info in PRIVACY_TAXONOMY.items():
        ex = " ".join(f"-{e}" for e in info["examples"])
        lines.append(f"  {cat}: {ex}")
    return "\n".join(lines)

def get_prompt_task3_dipa2():
    tax = get_taxonomy_string()
    return (
        'Task: Identify which privacy categories are violated in the following image '
        'based on the taxonomy below. Include all applicable categories.\n'
        f'{tax}\n'
        'Instructions:\n'
        '- Do not justify your answer.\n'
        '- Output ONLY a JSON object. No explanation, no markdown.\n'
        '- Output format: {"categories": ["Category Name 1", "Category Name 2"]}\n'
        '- Use exact category names from the taxonomy above.\n\n'
        'Now classify the image and output only the JSON.'
    )

PROMPT_TASK3 = get_prompt_task3_dipa2()
print("Prompt built (no Safe option, no numbered taxonomy).")
print("\nPrompt preview:")
print(PROMPT_TASK3[:500])

Prompt built (no Safe option, no numbered taxonomy).

Prompt preview:
Task: Identify which privacy categories are violated in the following image based on the taxonomy below. Include all applicable categories.
Taxonomy:
  Biometric Data: -face -fingerprints -audio -iris -gait
  Children Images: -school events -playgrounds
  Financial Information: -credit cards -checks -receipts
  HIPAA Data: -medical records -prescriptions -health devices -disabilities
  Legal Identifiers: -names -IDs -passports -addresses
  Digital Identifiers: -email -phone number -passwords -co


In [11]:
# ── Cell 10: Inference helpers & improved parsers ─────────────────
#
# INDEX_TO_CAT: built WITHOUT numbering in prompt (numbers removed to fix
# the known bug where models output "13" instead of "Location Identifiers").
# Still kept as a fallback in case model hallucinates numbers anyway.
#
# parse_task3 improvements over original:
#   FIX 1: Greedy regex r'\{.*\}' (non-greedy r'\{.*?\}' truncated JSON)
#   FIX 2: Handles 3 output formats:
#           Format 1 — {"categories": ["Biometric Data"]}        exact name
#           Format 2 — {"categories": ["5. Legal Identifiers"]}  strip "N." prefix
#           Format 3 — {"categories": ["13"]}                    resolve via INDEX_TO_CAT
#   FIX 3: Text-scan fallback uses \b word-boundary (prevents false positives
#           e.g. "no biometric data found" no longer matches "Biometric Data")
#   FIX 4: Explicit "safe" check before any category scan in fallback
#   FIX 5: Restricted to valid_categories — predictions outside active set discarded

INDEX_TO_CAT = {str(i): cat for i, cat in enumerate(PRIVACY_TAXONOMY.keys(), 1)}

def prepare_image(image_path, max_px=MAX_IMAGE_PX):
    img = Image.open(image_path).convert("RGB")
    if max(img.size) > max_px:
        img.thumbnail((max_px, max_px), Image.Resampling.LANCZOS)
    return img

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def call_model(image_path, prompt, temperature, max_new_tokens, seed=0):
    set_seed(seed)
    img = prepare_image(image_path)
    messages = [{"role":"user","content":[
        {"type":"image","image":img},
        {"type":"text","text":prompt}
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt", padding=True).to(model.device)
    do_sample = temperature > 0.05
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature if do_sample else None,
            do_sample=do_sample,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    response  = processor.batch_decode(generated, skip_special_tokens=True)[0].strip()
    del inputs, output_ids, generated
    torch.cuda.empty_cache()
    return response


def parse_task3(response, valid_categories=None):
    if valid_categories is None: valid_categories = VALID_CATEGORIES
    predicted = set()

    # ── Primary path: JSON parsing ───────────────────────────────
    try:
        # GREEDY regex — captures full JSON even with nested braces
        m = re.search(r'\{.*\}', response, re.DOTALL)
        if m:
            parsed = json.loads(m.group())
            cats = parsed.get("categories", [])
            if isinstance(cats, list):
                for c in cats:
                    c_str = str(c).strip()

                    # Safe signal (shouldn't appear but handle gracefully)
                    if c_str.lower() == "safe":
                        # In DIPA2 context, Safe means model predicted nothing —
                        # we treat as empty prediction (wrong, since all images private)
                        return set()

                    # Format 1: exact category name match
                    if c_str in valid_categories:
                        predicted.add(c_str)
                        continue

                    # Format 2: "N. Category Name" — strip numeric prefix
                    m2 = re.match(r'^\d+\.\s*(.+)$', c_str)
                    if m2:
                        name = m2.group(1).strip()
                        if name in valid_categories:
                            predicted.add(name)
                            continue
                        # case-insensitive fallback
                        for cat in valid_categories:
                            if cat.lower() == name.lower():
                                predicted.add(cat)
                                break
                        continue

                    # Format 3: pure digit index — resolve via INDEX_TO_CAT
                    if re.match(r'^\d+$', c_str):
                        resolved = INDEX_TO_CAT.get(c_str)
                        if resolved and resolved in valid_categories:
                            predicted.add(resolved)
                        continue

                    # Format 4: case-insensitive exact match (e.g. "biometric data")
                    for cat in valid_categories:
                        if cat.lower() == c_str.lower():
                            predicted.add(cat)
                            break

            if predicted:
                return predicted
    except Exception:
        pass

    # ── Fallback: word-boundary text scan ────────────────────────
    r = response.strip()
    # Explicit safe check first (model might output plain text "Safe")
    if re.search(r'\bsafe\b', r, re.IGNORECASE) and not predicted:
        return set()
    for cat in valid_categories:
        # \b word boundary prevents "no biometric data" matching "Biometric Data"
        if re.search(r'\b' + re.escape(cat) + r'\b', r, re.IGNORECASE):
            predicted.add(cat)
    return predicted


def majority_labels(all_runs, num_runs):
    counts = Counter(lbl for run in all_runs for lbl in run)
    return {cat for cat, cnt in counts.items() if cnt > num_runs / 2}


print("Inference helpers & improved parsers defined.")
print(f"INDEX_TO_CAT (fallback): {INDEX_TO_CAT}")

Inference helpers & improved parsers defined.
INDEX_TO_CAT (fallback): {'1': 'Biometric Data', '2': 'Children Images', '3': 'Financial Information', '4': 'HIPAA Data', '5': 'Legal Identifiers', '6': 'Digital Identifiers', '7': 'Personal Metadata (Demographics)', '8': 'GPS Data', '9': 'Vehicle Information', '10': 'Nudity', '11': 'Violent/Unlawful Actions', '12': 'Personal Context', '13': 'Location Identifiers', '14': 'Background Individuals'}


In [12]:
# ── Cell 11: Evaluation metrics ───────────────────────────────────

def evaluate_recognition(results, active_categories=None):
    if active_categories is None: active_categories = VALID_CATEGORIES
    # DIPA2: no Safe images — Safe category not scored
    has_safe = any(len(r["gt_labels"]) == 0 for r in results)
    all_cats = list(active_categories) + (["Safe"] if has_safe else [])
    category_metrics = {}
    for cat in all_cats:
        if cat == "Safe":
            y_true = [1 if len(r["gt_labels"])==0  else 0 for r in results]
            y_pred = [1 if len(r["pred_labels"])==0 else 0 for r in results]
        else:
            y_true = [1 if cat in r["gt_labels"]   else 0 for r in results]
            y_pred = [1 if cat in r["pred_labels"] else 0 for r in results]
        support = int(sum(y_true))
        if support == 0: continue
        p,r,f1,_ = precision_recall_fscore_support(
            y_true, y_pred, average="binary", pos_label=1, zero_division=0)
        category_metrics[cat] = {
            "precision": round(p*100,2),
            "recall":    round(r*100,2),
            "f1":        round(f1*100,2),
            "support":   support,
        }
    f1v = [m["f1"]        for m in category_metrics.values()]
    pv  = [m["precision"] for m in category_metrics.values()]
    rv  = [m["recall"]    for m in category_metrics.values()]
    return {
        "category_metrics":  category_metrics,
        "macro_f1":          round(np.mean(f1v), 2) if f1v else 0.0,
        "macro_precision":   round(np.mean(pv),  2) if pv  else 0.0,
        "macro_recall":      round(np.mean(rv),  2) if rv  else 0.0,
    }

print("Evaluation functions defined.")

Evaluation functions defined.


In [13]:
# ── Cell 12: Checkpoint helpers ───────────────────────────────────

CKPT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")

def save_checkpoint(results, temp, ckpt_dir=CKPT_DIR):
    os.makedirs(ckpt_dir, exist_ok=True)
    temp_str = str(temp).replace(".", "_")
    ts       = time.strftime("%Y%m%d_%H%M%S")
    path     = os.path.join(ckpt_dir, f"ckpt_task3_temp{temp_str}_{len(results)}imgs_{ts}.json")
    ser = []
    for r in results:
        rc = dict(r)
        if "gt_labels"   in rc: rc["gt_labels"]   = list(rc["gt_labels"])
        if "pred_labels" in rc: rc["pred_labels"] = list(rc["pred_labels"])
        ser.append(rc)
    with open(path, "w") as f:
        json.dump({
            "task": "task3", "temp": temp,
            "dataset": DATASET_NAME,
            "n_results": len(results),
            "results": ser
        }, f, indent=2)
    print(f"  ✓ checkpoint ({len(results)} imgs) → {path}")
    return path


def load_checkpoint(path):
    with open(path) as f:
        data = json.load(f)
    results = []
    for r in data["results"]:
        rc = dict(r)
        if "gt_labels"   in rc: rc["gt_labels"]   = set(rc["gt_labels"])
        if "pred_labels" in rc: rc["pred_labels"] = set(rc["pred_labels"])
        results.append(rc)
    print(f"Checkpoint loaded: {len(results)} results from {path}")
    return results


print("Checkpoint helpers defined.")
print(f"Checkpoint dir : {CKPT_DIR}")

Checkpoint helpers defined.
Checkpoint dir : /content/drive/MyDrive/DIPA2/qwen results/checkpoints


In [14]:
# ── Cell 13: Task 3 runner with checkpointing & resume ───────────

def run_task3(samples, temperature, valid_categories=None, resume_results=None):
    if valid_categories is None: valid_categories = DIPA2_ACTIVE_CATEGORIES

    # Resume: skip already-processed images
    results = list(resume_results) if resume_results else []
    done_ids = {r["id"] for r in results}
    remaining = [s for s in samples if s["id"] not in done_ids]

    if done_ids:
        print(f"  Resuming: {len(results)} done, {len(remaining)} remaining")

    t_start = time.time()
    for idx, sample in enumerate(remaining):
        run_labels  = []
        raw_outputs = []   # store raw model text for full audit trail

        for run in range(NUM_RUNS):
            raw    = call_model(
                sample["image_path"], PROMPT_TASK3,
                temperature, MAX_TOKENS["task3"], RUN_SEEDS[run]
            )
            parsed = parse_task3(raw, valid_categories)
            run_labels.append(parsed)
            raw_outputs.append(raw)

        # Majority vote across runs, then restrict to active categories
        final_labels = majority_labels(run_labels, NUM_RUNS) & valid_categories

        results.append({
            "id":          sample["id"],
            "image_path":  sample["image_path"],
            "gt_labels":   sample["taxonomy_labels"],   # set
            "pred_labels": final_labels,                # set
            "all_runs":    [list(r) for r in run_labels],
            "raw_outputs": raw_outputs,                 # full model text, all 3 runs
        })

        # Checkpoint
        if (len(results)) % CHECKPOINT_EVERY == 0:
            save_checkpoint(results, temperature)

        # Progress
        if (idx + 1) % 10 == 0 or idx == 0:
            el = time.time() - t_start
            print(f"  [{len(results):4d}/{len(samples)}]  "
                  f"elapsed {el:.0f}s  avg {el/(idx+1):.1f}s/img")

    return results, time.time() - t_start

print("Task runner defined.")

Task runner defined.


In [15]:
# ── Cell 14: MAIN PIPELINE (DIPA2 — Task 3 only) ─────────────────
print("\n" + "="*70)
print(f" MODEL   : {MODEL_ID}")
print(f" DATASET : {DATASET_NAME}  ({len(dataset)} images — all private, unanimous filter)")
print(f" TASKS   : Task 3 (Attribute Recognition) only")
print(f" ACTIVE CATS: {sorted(DIPA2_ACTIVE_CATEGORIES)}")
print("="*70 + "\n")

all_results    = {}
pipeline_start = time.time()

for temp in TEMPERATURES:
    temp_key = f"temp={temp}"
    print(f"\n{'─'*70}")
    print(f"TEMPERATURE: {temp}")
    print(f"{'─'*70}")
    all_results[temp_key] = {}

    # Handle resume
    resume_results = None
    if RESUME and RESUME_PATH and os.path.exists(RESUME_PATH):
        resume_results = load_checkpoint(RESUME_PATH)
        print(f"Resuming from checkpoint with {len(resume_results)} existing results")

    print(f"\n▶ Task 3: Attribute Recognition  (temp={temp})")
    r3, e3 = run_task3(
        dataset, temp,
        valid_categories=DIPA2_ACTIVE_CATEGORIES,
        resume_results=resume_results
    )
    m3 = evaluate_recognition(r3, active_categories=DIPA2_ACTIVE_CATEGORIES)
    all_results[temp_key]["task3"] = {
        "results": r3, "metrics": m3, "elapsed": e3
    }
    print(f"   Macro F1 : {m3['macro_f1']}%  |  Time: {e3:.0f}s  ({e3/max(len(r3),1):.1f}s/img)")
    print("   Per-category:")
    for cat, cm in sorted(m3["category_metrics"].items(), key=lambda x: x[1]["f1"], reverse=True):
        print(f"     {cat:<40} F1={cm['f1']:5.1f}%  P={cm['precision']:5.1f}%  R={cm['recall']:5.1f}%  n={cm['support']}")

# Mark best temperature
best_temp = max(
    all_results.keys(),
    key=lambda t: all_results[t]["task3"]["metrics"]["macro_f1"]
    if "task3" in all_results[t] else -1
)
all_results[best_temp]["task3"]["is_best"] = True

pe = time.time() - pipeline_start
print("\n" + "="*70)
print(" RESULTS SUMMARY — Task 3 (DIPA2, unanimous filter, 265 images)")
print("="*70)
print(f" {'Temp':>6}  {'Macro F1':>10}  {'Macro P':>10}  {'Macro R':>10}")
print("─"*70)
for temp_key, temp_data in all_results.items():
    if "task3" not in temp_data: continue
    m    = temp_data["task3"]["metrics"]
    best = " ★" if temp_data["task3"].get("is_best") else ""
    print(f" {temp_key.replace('temp=',''):>6}  {m['macro_f1']:>9.2f}%"
          f"  {m['macro_precision']:>9.2f}%  {m['macro_recall']:>9.2f}%{best}")
print("─"*70)
print(f" Total pipeline time: {pe:.0f}s  ({pe/60:.1f} min)")
print("="*70)


 MODEL   : Qwen/Qwen3-VL-8B-Instruct
 DATASET : DIPA2  (265 images — all private, unanimous filter)
 TASKS   : Task 3 (Attribute Recognition) only
 ACTIVE CATS: ['Background Individuals', 'Biometric Data', 'Digital Identifiers', 'Legal Identifiers', 'Location Identifiers', 'Personal Context', 'Vehicle Information', 'Violent/Unlawful Actions']


──────────────────────────────────────────────────────────────────────
TEMPERATURE: 0.1
──────────────────────────────────────────────────────────────────────

▶ Task 3: Attribute Recognition  (temp=0.1)
  [   1/265]  elapsed 7s  avg 6.8s/img
  ✓ checkpoint (10 imgs) → /content/drive/MyDrive/DIPA2/qwen results/checkpoints/ckpt_task3_temp0_1_10imgs_20260520_100254.json
  [  10/265]  elapsed 51s  avg 5.1s/img
  ✓ checkpoint (20 imgs) → /content/drive/MyDrive/DIPA2/qwen results/checkpoints/ckpt_task3_temp0_1_20imgs_20260520_100341.json
  [  20/265]  elapsed 97s  avg 4.8s/img
  ✓ checkpoint (30 imgs) → /content/drive/MyDrive/DIPA2/qwen results/chec

In [16]:
# ── Cell 15: Save results to Drive ───────────────────────────────
# JSON uses indent=2 → human-readable, auditable
# raw_outputs are included in every result entry for post-hoc parser verification

model_slug = MODEL_ID.replace("/","_").replace(".","_")
timestamp  = time.strftime("%Y%m%d_%H%M%S")

json_path = os.path.join(
    OUTPUT_DIR,
    f"{model_slug}_{DATASET_SLUG}_{timestamp}_results.json"
)

serializable = {}
for tk, td in all_results.items():
    serializable[tk] = {}
    for task, data in td.items():
        results_copy = []
        for r in data["results"]:
            rc = dict(r)
            # Convert sets to lists for JSON serialisation
            if "gt_labels"   in rc: rc["gt_labels"]   = sorted(list(rc["gt_labels"]))
            if "pred_labels" in rc: rc["pred_labels"] = sorted(list(rc["pred_labels"]))
            # all_runs and raw_outputs already lists
            results_copy.append(rc)
        serializable[tk][task] = {
            "metrics":    data["metrics"],
            "elapsed":    data["elapsed"],
            "is_best":    data.get("is_best", False),
            "n_images":   len(results_copy),
            "results":    results_copy,
        }

# Metadata block for reproducibility
serializable["_meta"] = {
    "model_id":       MODEL_ID,
    "dataset":        DATASET_NAME,
    "dataset_slug":   DATASET_SLUG,
    "n_images":       len(dataset),
    "selection":      "unanimous_4of4_annotators",
    "active_categories": sorted(DIPA2_ACTIVE_CATEGORIES),
    "temperatures":   TEMPERATURES,
    "num_runs":       NUM_RUNS,
    "seeds":          RUN_SEEDS,
    "timestamp":      timestamp,
}

with open(json_path, "w") as f:
    json.dump(serializable, f, indent=2)
print(f"JSON saved → {json_path}")

# Excel
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

def export_to_excel(all_results, output_path):
    wb = Workbook(); wb.remove(wb.active)
    HDR_FILL  = PatternFill("solid", fgColor="1F3864")
    SUB_FILL  = PatternFill("solid", fgColor="D6E4F0")
    BEST_FILL = PatternFill("solid", fgColor="E2EFDA")
    HDR_FONT  = Font(color="FFFFFF", bold=True, size=11)
    THIN      = Side(style="thin")
    BORDER    = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
    CENTER    = Alignment(horizontal="center", vertical="center", wrap_text=True)
    LEFT      = Alignment(horizontal="left",   vertical="center", wrap_text=True)
    def hdr(ws, row, col, val, width=None):
        c = ws.cell(row=row, column=col, value=val)
        c.fill=HDR_FILL; c.font=HDR_FONT; c.border=BORDER; c.alignment=CENTER
        if width: ws.column_dimensions[get_column_letter(col)].width = width
    def cell(ws, row, col, val, bold=False, fill=None, align=CENTER):
        c = ws.cell(row=row, column=col, value=val)
        c.font=Font(bold=bold); c.border=BORDER; c.alignment=align
        if fill: c.fill=fill
        return c

    # Sheet 1 — Attribute Recognition summary
    ws1 = wb.create_sheet("Attribute Recognition")
    for col,(label,w) in enumerate([
        ("Model",18),("Dataset",12),("Temperature",13),("Category",32),
        ("Precision (%)",13),("Recall (%)",13),("F1 (%)",11),("Support",9),("Best",7)
    ], 1):
        hdr(ws1,1,col,label,w)
    row = 2
    best_rec = None; best_f1v = -1
    for tk_, td_ in all_results.items():
        if not isinstance(td_, dict) or "task3" not in td_: continue
        m = td_["task3"]["metrics"]
        if m["macro_f1"] > best_f1v:
            best_f1v = m["macro_f1"]; best_rec = (tk_, m)
    if best_rec:
        tk_, m = best_rec
        for cat, cm in sorted(m["category_metrics"].items(), key=lambda x: x[1]["f1"], reverse=True):
            fill = SUB_FILL if cat == "Safe" else None
            cell(ws1,row,1,MODEL_NAME.split("/")[-1],bold=True,fill=fill,align=LEFT)
            cell(ws1,row,2,DATASET_NAME,fill=fill)
            cell(ws1,row,3,tk_,fill=fill)
            cell(ws1,row,4,cat,fill=fill,align=LEFT)
            cell(ws1,row,5,cm["precision"],fill=fill)
            cell(ws1,row,6,cm["recall"],fill=fill)
            cell(ws1,row,7,cm["f1"],bold=True,fill=fill)
            cell(ws1,row,8,cm["support"],fill=fill)
            cell(ws1,row,9,"★",fill=fill)
            row += 1
        row += 1
        cell(ws1,row,4,"MACRO AVERAGE",bold=True,fill=BEST_FILL,align=LEFT)
        cell(ws1,row,5,m["macro_precision"],bold=True,fill=BEST_FILL)
        cell(ws1,row,6,m["macro_recall"],bold=True,fill=BEST_FILL)
        cell(ws1,row,7,m["macro_f1"],bold=True,fill=BEST_FILL)

    # Sheet 2 — All temperatures comparison
    ws2 = wb.create_sheet("All Temperatures")
    for col,(label,w) in enumerate([
        ("Temperature",13),("Macro F1 (%)",13),("Macro P (%)",13),("Macro R (%)",13),("Best",7)
    ], 1):
        hdr(ws2,1,col,label,w)
    row = 2
    for tk_, td_ in all_results.items():
        if not isinstance(td_, dict) or "task3" not in td_: continue
        m = td_["task3"]["metrics"]
        is_best = td_["task3"].get("is_best", False)
        fill = BEST_FILL if is_best else None
        cell(ws2,row,1,tk_,fill=fill)
        cell(ws2,row,2,m["macro_f1"],bold=is_best,fill=fill)
        cell(ws2,row,3,m["macro_precision"],fill=fill)
        cell(ws2,row,4,m["macro_recall"],fill=fill)
        cell(ws2,row,5,"★" if is_best else "",fill=fill)
        row += 1

    # Sheet 3 — Raw predictions for audit
    ws3 = wb.create_sheet("Raw Predictions")
    for col,(label,w) in enumerate([
        ("Image ID",24),("Temperature",13),("GT Labels",30),("Pred Labels",30),
        ("Correct",9),("Run 1 Raw",40),("Run 2 Raw",40),("Run 3 Raw",40)
    ], 1):
        hdr(ws3,1,col,label,w)
    row = 2
    for tk_, td_ in all_results.items():
        if not isinstance(td_, dict) or "task3" not in td_: continue
        for r in td_["task3"]["results"]:
            gt   = str(sorted(r.get("gt_labels", [])))
            pred = str(sorted(r.get("pred_labels", [])))
            correct = "✓" if set(r.get("gt_labels",[])) == set(r.get("pred_labels",[])) else "✗"
            fill = BEST_FILL if correct == "✓" else None
            runs = r.get("raw_outputs", [])
            cell(ws3,row,1,r["id"],align=LEFT,fill=fill)
            cell(ws3,row,2,tk_,fill=fill)
            cell(ws3,row,3,gt,align=LEFT,fill=fill)
            cell(ws3,row,4,pred,align=LEFT,fill=fill)
            cell(ws3,row,5,correct,fill=fill)
            cell(ws3,row,6,runs[0][:200] if len(runs)>0 else "",align=LEFT,fill=fill)
            cell(ws3,row,7,runs[1][:200] if len(runs)>1 else "",align=LEFT,fill=fill)
            cell(ws3,row,8,runs[2][:200] if len(runs)>2 else "",align=LEFT,fill=fill)
            row += 1

    # Sheet 4 — Timing
    ws4 = wb.create_sheet("Timing")
    for col,(label,w) in enumerate([
        ("Temperature",13),("Images",9),("Total (s)",13),("Total (min)",14),("Avg/img (s)",13)
    ], 1):
        hdr(ws4,1,col,label,w)
    row = 2
    for tk_, td_ in all_results.items():
        if not isinstance(td_, dict) or "task3" not in td_: continue
        d = td_["task3"]; n = len(d["results"]); el = d["elapsed"]
        cell(ws4,row,1,tk_); cell(ws4,row,2,n)
        cell(ws4,row,3,round(el,1)); cell(ws4,row,4,round(el/60,2))
        cell(ws4,row,5,round(el/max(n,1),2)); row += 1

    wb.save(output_path)
    print(f"Excel saved → {output_path}")

excel_path = os.path.join(
    OUTPUT_DIR,
    f"{model_slug}_{DATASET_SLUG}_{timestamp}_results.xlsx"
)
export_to_excel(all_results, excel_path)
print("\nAll done.")

JSON saved → /content/drive/MyDrive/DIPA2/qwen results/Qwen_Qwen3-VL-8B-Instruct_dipa2_20260520_104048_results.json
Excel saved → /content/drive/MyDrive/DIPA2/qwen results/Qwen_Qwen3-VL-8B-Instruct_dipa2_20260520_104048_results.xlsx

All done.
